In [2]:
import pandas as pd
import numpy as np
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids
import default_risk.config as cfg


credit_card_df= pd.read_parquet(cfg.CLEANS_DIR / "credit_card_balance_train-cleaned.parquet")
credit_card_df.sort_values(["id_prev","months_balance"],inplace=True)

#auxiliar functions
def full_sorted_series_id(ids : pd.Series) -> pd.DataFrame :
    return recreate_and_sort_the_serie_given_ids(ids,credit_card_df,"id_prev","months_balance")
     

def full_sorted_series_row(rows : pd.DataFrame) -> pd.DataFrame :
    return recreate_and_sort_series_given_rows(rows,credit_card_df,"id_prev","months_balance")    

def time_window_credit_card(period_in_months, df_credit_card: pd.DataFrame) :
    last_period= df_credit_card [df_credit_card["months_balance"] > period_in_months]
    agg_last_period= last_period.groupby("id_curr").agg({
        "sk_dpd": ["mean","max"],
        "sk_dpd_def": ["mean","max"],
        "is_over_the_limit" : ["mean","sum"], 
        "inconsistency_gap": ["mean","max"],
        "desesperation_ratio" : ["max","mean"],
        "balance_limit_ratio" : ["min","max","mean"],
        "payment_ratio": ["min","mean"],
        "amt_payment_total_current": ["sum"],
        "amt_balance": ["sum","mean"],
        
    })
    period_in_months= period_in_months * -1

    agg_last_period.columns = [
    f"last_{period_in_months}_credit_card_{col[0]}_{col[1]}"
    for col in agg_last_period.columns
    ]
    agg_last_period= agg_last_period.reset_index()

    agg_last_period[f"last_{period_in_months}_credit_card_completion_ratio"] = np.where( agg_last_period[f"last_{period_in_months}_credit_card_amt_balance_sum"] > 0, agg_last_period[f"last_{period_in_months}_credit_card_amt_payment_total_current_sum"] / agg_last_period[f"last_{period_in_months}_credit_card_amt_balance_sum"], 1.0 ) 
    return agg_last_period
    

In [3]:
credit_card_df.head(10)

,id_prev,id_curr,months_balance,have_negative_balance,amt_balance,amt_negative_balance,diff_total_receivable_balance,amt_credit_limit_actual,amt_drawings_current,amt_drawings_atm_current,...,sk_dpd,sk_dpd_tecnical,sk_dpd_severe,sk_dpd_def,sk_dpd_def_tecnical,sk_dpd_def_severe,closing_month,non_closed_loan,potential_on_going_loan,incomplete_sequence
1148320,1000018,394447,-6,0,38879.145,0.0,-1336.5,45000,51042.645,13500.0,...,0,0,0,0,0,0,NaN,True,True,False
230081,1000018,394447,-5,0,40934.070,0.0,0.0,45000,2335.500,0.0,...,0,0,0,0,0,0,NaN,True,True,False
2135719,1000018,394447,-4,0,44360.505,0.0,0.0,45000,2032.560,0.0,...,0,0,0,0,0,0,NaN,True,True,False
1822742,1000018,394447,-3,0,113862.285,0.0,-4711.5,135000,69156.945,13500.0,...,0,0,0,0,0,0,NaN,True,True,False
1794265,1000018,394447,-2,0,136695.420,0.0,-670.5,135000,22827.330,0.0,...,0,0,0,0,0,0,NaN,True,True,False
1564529,1000030,361282,-8,0,0.000,0.0,0.0,45000,0.000,0.0,...,0,0,0,0,0,0,NaN,True,True,False
799031,1000030,361282,-7,0,15583.635,0.0,-445.5,45000,31105.755,4500.0,...,0,0,0,0,0,0,NaN,True,True,False
1822764,1000030,361282,-6,0,33784.740,0.0,0.0,45000,20212.650,0.0,...,0,0,0,0,0,0,NaN,True,True,False
1833045,1000030,361282,-5,0,36885.285,0.0,0.0,45000,6368.850,0.0,...,0,0,0,0,0,0,NaN,True,True,False
1432683,1000030,361282,-4,0,59188.050,0.0,0.0,135000,25312.050,0.0,...,0,0,0,0,0,0,NaN,True,True,False


In [4]:
credit_card_df["raw_lenght"]= credit_card_df.groupby("id_prev").transform("size") 
credit_card_df["next_mature_cum"] = credit_card_df.groupby("id_prev")["cnt_instalment_mature_cum"].shift(-1)
credit_card_df["month_with_activity"] = (credit_card_df["next_mature_cum"] > 0) & (credit_card_df["amt_balance"] > 0)
credit_card_df["is_over_the_limit"] = (credit_card_df["amt_credit_limit_actual"] >  credit_card_df["amt_balance"] ).astype(int)
credit_card_df["balance_limit_ratio"] = np.where((credit_card_df["amt_credit_limit_actual"] != 0) & (credit_card_df["amt_balance"] > 0) ,credit_card_df["amt_balance"]  / credit_card_df["amt_credit_limit_actual"],np.nan)
credit_card_df["payment_ratio"] = np.where((credit_card_df["amt_payment_total_current"] != 0) & (credit_card_df["amt_inst_min_regularity"] > 0) ,credit_card_df["amt_payment_total_current"]  / credit_card_df["amt_inst_min_regularity"],np.nan)
credit_card_df["desesperation_ratio"] = np.where((credit_card_df["amt_drawings_current"] != 0) & (credit_card_df["amt_drawings_atm_current"] > 0),credit_card_df["amt_drawings_atm_current"]  / credit_card_df["amt_drawings_current"],np.nan)
                                                                     
credit_card_df["name_contract_status"]= credit_card_df["name_contract_status"].str.lower()
credit_card_df= pd.get_dummies(credit_card_df,columns= ["name_contract_status"])

credit_card_df["potential_on_going_loan"]= credit_card_df["potential_on_going_loan"].astype(int)



with_dpd= credit_card_df[(credit_card_df["sk_dpd"] != 0)  & (credit_card_df["sk_dpd_def"])] 
months_since_dpd= with_dpd.groupby("id_prev")["months_balance"].transform("max")
credit_card_df["months_since_dpd"]= credit_card_df["id_prev"].map(months_since_dpd)


In [5]:
credit_card_agg_df= credit_card_df.groupby("id_prev").agg({
    "potential_on_going_loan" : ["first"],
    "incomplete_sequence" : ["first"],
    "closing_month": ["first"],
    "raw_lenght" : ["first"],
    "first_expected_payment_month": ["first"],
    "months_since_dpd": ["first"],

    "amt_balance": ["min","max","mean","std"],
    "balance_limit_ratio": ["min","max","mean","std"],
    "payment_ratio": ["min","max","mean","std"],
    "desesperation_ratio" : ["min","max","mean","std"],
    "amt_negative_balance": ["min","mean","std"],
    "diff_total_receivable_balance": ["min","mean","max","std"],
    "amt_credit_limit_actual": ["min","mean","max","std"],
    "amt_drawings_atm_current": ["mean","max","std","sum"],
    "amt_drawings_other_current": ["mean","max","std","sum"],
    "amt_drawings_pos_current": ["mean","max","std","sum"],
    "amt_inst_min_regularity": ["min","mean","max","std","sum"],
    "inconsistency_gap": ["mean","max","sum"],
    "diff_payment_current_total": ["mean","max","sum"],
    "amt_payment_total_current": ["min","mean","max","std","sum"],
    "amt_recivable_principal": ["mean","max","std","sum"],
    "amt_recivable": ["mean","max","std","sum"],
    "cnt_drawings_current": ["mean","max","sum"],
    "cnt_drawings_atm_current": ["mean","max","sum"],
    "cnt_drawings_other_current": ["mean","max","sum"],
    "cnt_drawings_pos_current": ["mean","max","sum"],
    "cnt_instalment_mature_cum": ["max"],

    "months_balance": ["max", "min"],
  

    #day_past_due
    "sk_dpd": ["mean","sum","max"],
    "sk_dpd_def": ["mean","sum","max"],

    #categoricals
    "sk_dpd_tecnical": ["mean", "sum"],
    "sk_dpd_severe": ["mean", "sum"],
    "sk_dpd_def_tecnical": ["mean", "sum"],
    "sk_dpd_def_severe": ["mean", "sum"],
    "name_contract_status_active": ["mean", "sum"],
    "name_contract_status_completed": ["mean", "sum"],
    "have_negative_balance": ["mean", "sum"],
    "cnt_drawings_are_present": ["mean", "sum"],
    "month_with_activity" : ["mean", "sum"], 
    "name_contract_status_demand" : ["mean", "sum"], 
    "is_over_the_limit" : ["mean", "sum"], 
})

credit_card_agg_df.columns = [
    f"credit_card_{col[0]}" if col[1] == "first" else f"credit_card_{col[0]}_{col[1]}"
    for col in credit_card_agg_df.columns
]

credit_card_agg_df.to_parquet(cfg.PROCESSED_DIR / "credit_card_agg.parquet")

In [ ]:
last_six_months= time_window_credit_card(-6,credit_card_df)
last_six_months.to_parquet(cfg.PROCESSED_DIR / "last_six_months_agg.parquet")

: 